# Modular semi-Mapper pipeline — CTN-0051

This notebook is a thin driver over the `mapper` package. All logic lives in the package modules; here you only **tune parameters and call the pipeline**.

Pipeline: `data -> distance -> lens -> cover -> graph -> layout -> viz`.

Three lenses are available:
- **feature**    — a data column (original behaviour, e.g. `attendance_density_w8`)
- **centrality** — graph centrality on the proximity graph (degree, betweenness, ...)
- **density**    — local density in feature space (knn, ball, kde)

## 1. Parameters — tune here

In [30]:
from mapper import MapperParams, run_pipeline, visualise
from mapper import diagnostics as dg
import numpy as np

params = MapperParams(
    PRE_TRIAL_CSV="../../data/clean-data/pre-trial.csv",
    TARGET_CSV   ="../../data/clean-data/retention_tier.csv",
    W8_CSV       ="../../data/clean-data/features_w8.csv",
    W12_CSV      ="../../data/clean-data/features_w12.csv",

    # --- proximity ---
    EPSILON=0.56,
    METRIC ="cosine",        # euclidean | cosine | manhattan | minkowski
    MINKOWSKI_P= np.inf,            # only for minkowski

    # --- LENS: pick one of the three ---
    LENS_KIND="feature",        # "feature" | "centrality" | "density"
    FEATURE_LENS_COL  = "detox_los",   # feature lens: attendance_density_w8 | detox_los 
    CENTRALITY_MEASURE="betweenness",             # centrality lens
    DENSITY_METHOD    ="knn",                     # density lens
    DENSITY_K         =10,

    # --- COVER / BINNING (tunable) ---
    COVER_MODE ="uniform",      # "uniform" (N_INTERVALS+OVERLAP) or "edges" (BIN_EDGES)
    N_INTERVALS= 60,
    OVERLAP    =0.7,
    # For explicit clinical bins instead, use:
    # COVER_MODE="edges", BIN_EDGES=[0,3,7,14,21], BIN_LABELS=["0-3","3-7","7-14","14+"]

    PIECEWISE_SEGMENTS=[
        (0.0, 3, 10),    # dense cover in the first half
        (3, 6, 10),
        (6, 9, 10),
        (9, 15, 10),
        (15, 21, 5),
        (15, 45, 5)   # sparse cover in the second half
    ],


    # --- edge rule for the displayed graph ---
    EDGE_RULE  ="cover",        # "cover" (share a set) | "gap" | "none"
    MAX_BIN_GAP=1,

    # --- layout & colour ---
    LAYOUT  ="spring", 
    SPRING_K = 1,              # spring | spectral | pca
    COLOR_BY="tier",            # "tier" or "lens"
)
print(params.summary())

ε=0.56 | metric=cosine | lens=feature=detox_los | cover=uniform(60x70%) | edge_rule=cover | layout=spring


## 2. Run the pipeline

In [31]:
result = run_pipeline(params)   # prints stage-by-stage summaries

Patients        : 554
Feature matrix  : (554, 57)
Missing values  : 0

Retention tier distribution:
retention_tier
1    164
2    108
3     61
4    221 

Distance matrix : (554, 554)
Distance range  : [0.000, 1.692]
Percentiles:
   10th : 0.704
   25th : 0.859
   50th : 1.014
   75th : 1.154
   90th : 1.271
With ε = 0.56:
  Edges before pruning : 5023
  Edge density         : 3.3% 

[feature] detox_los: min=1.000 median=6.000 max=40.000 (n_valid=554) 

Cover mode: uniform  |  60 sets

  Set  0 [1,3.17)        :  69 patients
  Set  1 [1.65,3.82)     :  63 patients
  Set  2 [2.3,4.47)      :  84 patients
  Set  3 [2.95,5.12)     : 173 patients
  Set  4 [3.6,5.77)      : 133 patients
  Set  5 [4.25,6.42)     : 183 patients
  Set  6 [4.9,7.07)      : 252 patients
  Set  7 [5.55,7.72)     : 163 patients
  Set  8 [6.2,8.37)      : 115 patients
  Set  9 [6.85,9.02)     : 143 patients
  Set 10 [7.5,9.67)      :  74 patients
  Set 11 [8.15,10.3)     :  42 patients
  Set 12 [8.8,11)        :  42 

## 3. Interactive Bokeh graph

In [32]:
from bokeh.io import output_notebook
output_notebook()
visualise(result)

Loading BokehJS ...

figure(id='p1896', ...)

In [ ]:
from bokeh.io import output_file, save
import networkx as nx
import copy

# Get the figure without rendering it in the notebook
fig = visualise(result, render=False)

# Save Bokeh HTML
output_file("../../graphs/w12/att_density12_a.html", title="Mapper Graph – Week 12 Attendance Density")
save(fig)

# GraphML only supports scalar types — convert any list attributes to strings
G_export = copy.deepcopy(result.graph)
for _, data in G_export.nodes(data=True):
    for k, v in list(data.items()):
        if isinstance(v, list):
            data[k] = ",".join(map(str, v))
for _, _, data in G_export.edges(data=True):
    for k, v in list(data.items()):
        if isinstance(v, list):
            data[k] = ",".join(map(str, v))

nx.write_graphml(G_export, "../../graphs/w12/att_density12_a.graphml")
# Reload later:
G = nx.read_graphml("../../graphs/w12/att_density12_a.graphml")